In [50]:
import numpy as np

data = [
    [12.0, 1.5, 1, 'Wine'],
    [5.0, 2.0, 0, 'Beer'],
    [40.0, 0.0, 1, 'Whiskey'],
    [13.5, 1.2, 1, 'Wine'],
    [4.5, 1.8, 0, 'Beer'],
    [38.0, 0.1, 1, 'Whiskey'],
    [11.5, 1.7, 1, 'Wine'],
    [5.5, 2.3, 0, 'Beer']
]

# encode labels
label_map = {'Wine': 0, 'Beer': 1, 'Whiskey': 2}

X = []
y = []

for row in data:
    X.append(row[:3])
    y.append(label_map[row[3]])

X = np.array(X, dtype=float)
y = np.array(y, dtype=int)


In [51]:
def gini(y):
    classes, counts = np.unique(y, return_counts=True)
    probs = counts / len(y)
    return 1 - np.sum(probs ** 2)

In [52]:
def split(X, y, feature, threshold):
    left_mask = X[:, feature] < threshold
    right_mask = X[:, feature] >= threshold

    X_left = X[left_mask]
    y_left = y[left_mask]

    X_right = X[right_mask]
    y_right = y[right_mask]

    return X_left, X_right, y_left, y_right



In [53]:

def best_split(X, y):
    best_gini = float('inf')
    best_feature = None
    best_threshold = None

    n_samples, n_features = X.shape

    for feature in range(n_features):
        thresholds = np.unique(X[:, feature])

        for threshold in thresholds:
            X_left, X_right, y_left, y_right = split(X, y, feature, threshold)

            # skip bad splits
            if len(y_left) == 0 or len(y_right) == 0:
                continue

            # compute gini
            gini_left = gini(y_left)
            gini_right = gini(y_right)

            weighted_gini = (
                len(y_left) * gini_left + len(y_right) * gini_right
            ) / n_samples

            # update best split
            if weighted_gini < best_gini:
                best_gini = weighted_gini
                best_feature = feature
                best_threshold = threshold

    return best_feature, best_threshold, best_gini



In [54]:

class Node:
    def __init__(self, feature_index=None, threshold=None,
                 left=None, right=None, value=None):

        self.feature_index = feature_index  # which feature
        self.threshold = threshold          # split value
        self.left = left                    # left subtree
        self.right = right                  # right subtree
        self.value = value                  # leaf value


def build_tree(X, y, depth=0, max_depth=5, min_samples=1):

    # STOP conditions
    if (len(set(y)) == 1 or
        depth >= max_depth or
        len(y) < min_samples):

        # majority class
        leaf_value = max(set(y), key=list(y).count)
        return Node(value=leaf_value)

    # find best split
    feature, threshold, _ = best_split(X, y)

    # if no split possible
    if feature is None:
        leaf_value = max(set(y), key=list(y).count)
        return Node(value=leaf_value)

    # split data
    X_left, X_right, y_left, y_right = split(X, y, feature, threshold)

    # recursive calls
    left_child = build_tree(X_left, y_left, depth + 1, max_depth, min_samples)
    right_child = build_tree(X_right, y_right, depth + 1, max_depth, min_samples)

    # return decision node
    return Node(feature_index=feature,
                threshold=threshold,
                left=left_child,
                right=right_child)



In [55]:

def predict(node, x):

    # if Nodeleaf  then return value
    if node.value is not None:
        return node.value

    # else
    if x[node.feature_index] < node.threshold:
        return predict(node.left, x)
    else:
        return predict(node.right, x)


In [56]:
tree = build_tree(X, y)

test_data = np.array([
    [6.0, 2.1, 0],   # Beer
    [39.0, 0.05, 1], # Whiskey
    [13.0, 1.3, 1]   # Wine
])

# reverse mapping
r_map = {0: 'Wine', 1: 'Beer', 2: 'Whiskey'}

predictions = []

for sample in test_data:
    pred = predict(tree, sample)
    predictions.append(r_map[pred])

print("Predictions:", predictions)

Predictions: ['Beer', 'Whiskey', 'Wine']
